# Sentiment Analysis using Random Forest

In this notebook, we will train a Random Forest to read movie reviews and predict if they are positive or negative.
We will break this down step-by-step so it is easy to understand!

In [6]:
# First, install the required libraries
!pip install pandas scikit-learn sentence-transformers tqdm scipy

### Step 1: Import Libraries

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

tqdm.pandas()

### Step 2: Load the Data
Here we load the cleaned movie reviews from our CSV file.

In [8]:
print("Loading dataset...")
df = pd.read_csv('../Data/IMDB_Dataset_Cleaned_Advanced.csv', encoding='latin-1')

# Remove any empty rows
df = df.dropna(subset=['review'])

# Convert 'positive' to 1 and 'negative' to 0
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Loaded {df.shape[0]} reviews.")
df.head(3)

Loading dataset...
Loaded 50000 reviews.


,review,sentiment
0,reviewer mention watch episode hook right exac...,1
1,wonderful little production technique unassumi...,1
2,think wonderful way spend time hot summer week...,1


### Step 3: Extract Keyword Features (TF-IDF)
This step counts the most important words and phrases (like "good", "bad movie") and creates 10,000 features for each review.

In [9]:
print("Extracting top 10,000 TF-IDF keyword features...")
reviews_list = df['review'].tolist()

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 3))
X_tfidf = tfidf.fit_transform(reviews_list).toarray() 

print("Keyword features extracted!")

Extracting top 10,000 TF-IDF keyword features...
Keyword features extracted!


### Step 4: Extract Semantic Embeddings (Runs on GPU)
This step uses a powerful AI model to understand the deep meaning of the sentences. It turns each review into 1,024 numbers.

In [10]:
print("Loading SentenceTransformer model...")
model = SentenceTransformer('BAAI/bge-large-en-v1.5')

print("Encoding text into 1024-dimensional dense vectors...")
X_dense = model.encode(reviews_list, show_progress_bar=True)

print("Embeddings extracted!")

Loading SentenceTransformer model...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 475.47it/s]


Encoding text into 1024-dimensional dense vectors...


Batches: 100%|██████████| 1563/1563 [20:44<00:00,  1.26it/s]


Embeddings extracted!


### Step 5: Combine All Features
Now we combine the keyword features and the semantic features together into one massive table.

In [11]:
print("Combining both feature sets...")
X = np.hstack([X_dense, X_tfidf])
y = df['sentiment'].values

print(f"Final Dataset Shape: {X.shape}")

Combining both feature sets...
Final Dataset Shape: (50000, 11024)


### Step 6: Split into Train and Test Sets
We will use 80% of the data to train the model, and hide 20% to test it later.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} samples.")
print(f"Testing on {len(X_test)} samples.")

Training on 40000 samples.
Testing on 10000 samples.


### Step 7: Train the Random Forest
Now we train the Random Forest Classifier on our training data. This will take a few minutes.

In [17]:
print("Training 'Strictly Regulated' Random Forest...")
# We add strict constraints (max_depth, min_samples) to completely prevent overfitting!
rf = RandomForestClassifier(
    n_estimators=500, 
    max_depth=20,          # STRICT: Stops trees from growing too deep and memorizing data
    min_samples_split=50,  # STRICT: Requires 50 samples to split a node
    min_samples_leaf=20,   # STRICT: Requires 20 samples to form a leaf
    max_features='sqrt',
    random_state=42,
    n_jobs=-1 
)

rf.fit(X_train, y_train)

print("Training complete!")

Training 'Strictly Regulated' Random Forest...
Training complete!


### Step 8: Calculate Training Accuracy
Let's see how well the model predicts the data it was trained on.

In [18]:
train_preds = rf.predict(X_train)
train_acc = accuracy_score(y_train, train_preds)

print(f"TRAINING ACCURACY: {train_acc * 100:.2f}%")

TRAINING ACCURACY: 93.08%


### Step 9: Calculate Testing Accuracy
Let's see how well the model predicts unseen test data. This is the most important metric!

In [19]:
test_preds = rf.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)

print(f"TESTING ACCURACY: {test_acc * 100:.2f}%")

TESTING ACCURACY: 87.01%


### Step 10: Detailed Classification Report (Precision, Recall, F1-Score)
Finally, we look at the overall advanced metrics to see exactly how well the model handles positive vs negative reviews.

In [20]:
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, test_preds))


--- Detailed Classification Report ---
              precision    recall  f1-score   support

           0       0.89      0.85      0.87      4961
           1       0.86      0.89      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000

